In [4]:
import pandas as pd
# --- 1. Cargar dataset ---
# Ruta gestionada por DVC
df = pd.read_csv(r"C:\Users\Elenita\Desktop\proyectofinal4\dp261-g3\data\processed\04_default_credit_features.csv")


df.head()

,ID,LIMIT_BAL,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,...,payment_ratio,late_payments,bill_trend,pay_trend,age_group,credit_level,risk_score,ability_to_pay,log_limit,log_avg_bill
0,1,20000,24.0,1.5,1.5,-1.0,-1.0,-2.0,-2.0,3913.0,...,0.089364,2,-3913.0,0.0,young,"(9485.0, 113000.0]",40000,689.0,9.903538,7.158514
1,2,120000,26.0,-1.0,1.5,0.0,0.0,0.0,1.5,2682.0,...,0.292689,6,579.0,2000.0,adult,"(113000.0, 216000.0]",720000,2000.0,11.695255,7.954080
2,3,90000,34.0,0.0,0.0,0.0,0.0,0.0,0.0,29239.0,...,0.108382,6,-13690.0,3482.0,adult,"(9485.0, 113000.0]",540000,4018.0,11.407576,9.737620
3,4,50000,37.0,0.0,0.0,0.0,0.0,0.0,0.0,46990.0,...,0.036258,6,-17443.0,-1000.0,middle,"(9485.0, 113000.0]",300000,5219.0,10.819798,10.559884
4,5,50000,57.0,-1.0,0.0,-1.0,0.0,0.0,0.0,8617.0,...,0.307453,6,10514.0,-1321.0,senior,"(9485.0, 113000.0]",300000,23250.5,10.819798,9.810504


In [5]:
#BALANCEO DE CLASES

# --- Librerías ---
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from imblearn.over_sampling import SMOTE, RandomOverSampler, ADASYN
from imblearn.under_sampling import RandomUnderSampler, NearMiss
from imblearn.combine import SMOTEENN, SMOTETomek
import os
from sklearn.preprocessing import StandardScaler # Import StandardScaler


# --- 2. Variables predictoras y target ---
X = df.drop("default payment next month", axis=1)
y = df["default payment next month"]

# Identify categorical columns for one-hot encoding
categorical_cols = X.select_dtypes(include=['object', 'bool']).columns

# Apply one-hot encoding to categorical columns in X
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# --- 3. Verificar ratio de clases ---
print("Distribución original:", y.value_counts(normalize=True))

# --- 4. Train/Test split con stratify=y ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# --- 5. Definir técnicas de balanceo ---
samplers = {
    "SMOTE": SMOTE(random_state=42),
    "RandomOverSampler": RandomOverSampler(random_state=42),
    "ADASYN": ADASYN(random_state=42),
    "RandomUnderSampler": RandomUnderSampler(random_state=42),
    "NearMiss": NearMiss(),
    "SMOTEENN": SMOTEENN(random_state=42),
    "SMOTETomek": SMOTETomek(random_state=42)
}

# --- 6. Modelo baseline ---
model = LogisticRegression(max_iter=5000)

# --- 7. Loop de balanceo + evaluación ---
results = []

for name, sampler in samplers.items():
    # Balancear SOLO el train
    X_res, y_res = sampler.fit_resample(X_train, y_train)

    # Scale the data for Logistic Regression
    scaler = StandardScaler()
    X_res_scaled = scaler.fit_transform(X_res)
    X_test_scaled = scaler.transform(X_test)

    # Entrenar modelo
    model.fit(X_res_scaled, y_res)

    # Evaluar en test desbalanceado
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:,1]

    report = classification_report(y_test, y_pred, output_dict=True)
    auc = roc_auc_score(y_test, y_proba)
    auc_pr = average_precision_score(y_test, y_proba)

    results.append({
        "Balanceo": name,
        "Recall clase 1": report["1"]["recall"],
        "F1 clase 1": report["1"]["f1-score"],
        "ROC-AUC": auc,
        "AUC-PR": auc_pr
    })

# --- 8. Tabla comparativa ---
results_df = pd.DataFrame(results)
print(results_df)



C:\Users\Elenita\AppData\Local\Temp\ipykernel_47460\2673050026.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object', 'bool']).columns


Distribución original: default payment next month
0    0.7788
1    0.2212
Name: proportion, dtype: float64
             Balanceo  Recall clase 1  F1 clase 1   ROC-AUC    AUC-PR
0               SMOTE        0.374529    0.449164  0.722826  0.474407
1   RandomOverSampler        0.642803    0.496363  0.737593  0.482567
2              ADASYN        0.364732    0.443426  0.719579  0.470372
3  RandomUnderSampler        0.648832    0.501748  0.737379  0.475042
4            NearMiss        0.642803    0.377935  0.610749  0.328949
5            SMOTEENN        0.543331    0.485849  0.725022  0.468543
6          SMOTETomek        0.381311    0.454015  0.723809  0.473486


In [6]:
# Recall clase 1 = qué tan bien detecta la clase minoritaria (los clientes que sí incumplen).
# F1 clase 1 = equilibrio entre precisión y recall para la clase minoritaria.
# ROC-AUC = capacidad general del modelo para distinguir entre clases.
# AUC-PR = rendimiento específico en la clase positiva (más sensible en datasets desbalanceados).




In [7]:
# RandomOverSampler y RandomUnderSampler destacan con recall clase 1 ≈ 0.64–0.65 y F1 alrededor de 0.50, lo que significa que capturan mejor los casos positivos (default).
# SMOTE y SMOTETomek tienen recall más bajo (~0.37–0.38), aunque mantienen un ROC-AUC decente (~0.72).
# NearMiss logra buen recall (~0.64) pero su F1 y AUC-PR son más bajos, lo que indica que sacrifica precisión.
# SMOTEENN queda en un punto intermedio, con recall moderado (~0.54) y F1 cercano a 0.48.
# La prioridad es detectar la mayor cantidad de defaults (recall alto), RandomOverSampler y RandomUnderSampler son las mejores opciones de tecnicas de balance de datos.




In [8]:
# --- 9. - Exportar datasets balanceados de las mejores técnicas ---

best_samplers = ["RandomOverSampler", "RandomUnderSampler"]

for name in best_samplers:
    sampler = samplers[name]
    X_res, y_res = sampler.fit_resample(X_train, y_train)

    df_res = pd.DataFrame(X_res, columns=X_train.columns)
    df_res["default payment next month"] = y_res

    filename = fr"C:\Users\Elenita\Desktop\proyectofinal4\dp261-g3\data\processed\04_default_credit_balanced_{name}.csv"
    df_res.to_csv(filename, index=False)
    print(f"✅ Dataset balanceado con {name} guardado en {filename}")

✅ Dataset balanceado con RandomOverSampler guardado en C:\Users\Elenita\Desktop\proyectofinal4\dp261-g3\data\processed\04_default_credit_balanced_RandomOverSampler.csv
✅ Dataset balanceado con RandomUnderSampler guardado en C:\Users\Elenita\Desktop\proyectofinal4\dp261-g3\data\processed\04_default_credit_balanced_RandomUnderSampler.csv
